# Quant Data Platform — Data Inspection Notebook

Use this notebook to inspect local CSV, Parquet, DuckDB, Postgres metadata, and GCS data.

Recommended placement in your repo:

```text
notebook/quant_data_inspection.ipynb
```

Recommended VS Code kernel:

```text
Python (.venv - quant platform)
```


## 0. Setup

In [1]:
from pathlib import Path
import os
from io import StringIO

import pandas as pd
import duckdb
from dotenv import load_dotenv

try:
    import psycopg
except ImportError:
    psycopg = None

try:
    from google.cloud import storage
except ImportError:
    storage = None

# If notebook is opened from project root, this is correct.
# If notebook is opened from notebook/, use: PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

print("PROJECT_ROOT:", PROJECT_ROOT)
print(".env exists:", (PROJECT_ROOT / ".env").exists())
print("Python cwd:", Path.cwd())


PROJECT_ROOT: c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform
.env exists: True
Python cwd: c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\notebook


In [2]:
def show_df(df: pd.DataFrame, n: int = 10, title: str | None = None):
    if title:
        print(f"\n{title}")
        print("-" * len(title))
    print("shape:", df.shape)
    display(df.head(n))
    return df

def path_info(path: str | Path):
    p = Path(path)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    print("path:", p)
    print("exists:", p.exists())
    if p.exists() and p.is_file():
        print("size_mb:", round(p.stat().st_size / 1024 / 1024, 3))
    if p.exists() and p.is_dir():
        children = list(p.iterdir())
        print("n_children:", len(children))
        for child in children[:20]:
            print(" -", child.name)


## 1. Inspect CSV

In [3]:
csv_path = PROJECT_ROOT / "data/ods/source=tiingo/dataset=supported_tickers/supported_tickers.csv"
path_info(csv_path)

if csv_path.exists():
    df_csv = pd.read_csv(csv_path)
    show_df(df_csv, title="supported_tickers.csv")
    display(df_csv.dtypes.to_frame("dtype"))


path: c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\ods\source=tiingo\dataset=supported_tickers\supported_tickers.csv
exists: True
size_mb: 5.236

supported_tickers.csv
---------------------
shape: (129528, 6)


,ticker,exchange,assetType,priceCurrency,startDate,endDate
0,-P-H,NYSE,Stock,USD,NaN,NaN
1,-P-HIZ,NASDAQ,Stock,USD,2023-08-30,2025-09-29
2,000001,SHE,Stock,CNY,2007-01-04,2026-06-01
3,000002,SHE,Stock,CNY,2007-01-04,2026-06-01
4,000003,SHE,Stock,CNY,NaN,NaN
5,000004,SHE,Stock,CNY,2007-08-31,2026-04-27
6,000005,SHE,Stock,CNY,2007-08-31,2024-03-05
7,000006,SHE,Stock,CNY,2007-01-04,2026-06-01
8,000007,SHE,Stock,CNY,2007-08-31,2026-06-01
9,000008,SHE,Stock,CNY,2007-01-04,2026-06-01


,dtype
ticker,object
exchange,object
assetType,object
priceCurrency,object
startDate,object
endDate,object


In [4]:
audit_csv_path = PROJECT_ROOT / "reports/backfill_audit/pilot_500/symbol_coverage.csv"
path_info(audit_csv_path)

if audit_csv_path.exists():
    df_audit = pd.read_csv(audit_csv_path)
    show_df(df_audit, title="pilot_500 symbol_coverage.csv")
    display(df_audit.dtypes.to_frame("dtype"))


path: c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\reports\backfill_audit\pilot_500\symbol_coverage.csv
exists: True
size_mb: 0.154

pilot_500 symbol_coverage.csv
-----------------------------
shape: (500, 26)


,task_id,task_list_name,source,dataset_name,security_id,ticker,requested_start_date,requested_end_date,priority,task_status,created_at,last_successful_date,metadata_status,metadata_attempt_count,metadata_last_error_message,metadata_updated_at,n_rows,min_date,max_date,invalid_adj_close_rows,invalid_volume_rows,status,attempt_count,last_error_message,updated_at,has_dwd_rows
0,pilot_500:tiingo:equity_price_daily:AACIU:2019-01-01:2026-06-02,pilot_500,tiingo,equity_price_daily,tiingo:AACIU,AACIU,2019-01-01,2026-06-02,1,pending,2026-06-02T23:21:24.443361+00:00,2026-06-02,success,1,NaN,2026-06-07 17:48:32.874502+00:00,73,2026-02-18,2026-06-02,0.0,0.0,success,1,NaN,2026-06-07 17:48:32.874502+00:00,True
1,pilot_500:tiingo:equity_price_daily:AAPL:2019-01-01:2026-06-02,pilot_500,tiingo,equity_price_daily,tiingo:AAPL,AAPL,2019-01-01,2026-06-02,1,pending,2026-06-02T23:21:24.443361+00:00,2026-06-02,success,1,NaN,2026-06-04 11:08:37.217701+00:00,1864,2019-01-02,2026-06-02,0.0,0.0,success,1,NaN,2026-06-04 11:08:37.217701+00:00,True
2,pilot_500:tiingo:equity_price_daily:ABOS:2019-01-01:2026-06-02,pilot_500,tiingo,equity_price_daily,tiingo:ABOS,ABOS,2019-01-01,2026-06-02,1,pending,2026-06-02T23:21:24.443361+00:00,2026-06-02,success,1,NaN,2026-06-07 17:48:33.328582+00:00,1235,2021-07-01,2026-06-02,0.0,0.0,success,1,NaN,2026-06-07 17:48:33.328582+00:00,True
3,pilot_500:tiingo:equity_price_daily:ACACU:2019-01-01:2026-06-02,pilot_500,tiingo,equity_price_daily,tiingo:ACACU,ACACU,2019-01-01,2026-06-02,1,pending,2026-06-02T23:21:24.443361+00:00,2024-09-26,success,1,NaN,2026-06-07 17:48:33.826170+00:00,576,2022-06-10,2024-09-26,0.0,0.0,success,1,NaN,2026-06-07 17:48:33.826170+00:00,True
4,pilot_500:tiingo:equity_price_daily:ACCO:2019-01-01:2026-06-02,pilot_500,tiingo,equity_price_daily,tiingo:ACCO,ACCO,2019-01-01,2026-06-02,1,pending,2026-06-02T23:21:24.443361+00:00,2026-06-02,success,1,NaN,2026-06-07 17:48:34.263666+00:00,1864,2019-01-02,2026-06-02,0.0,0.0,success,1,NaN,2026-06-07 17:48:34.263666+00:00,True
5,pilot_500:tiingo:equity_price_daily:ACEVU:2019-01-01:2026-06-02,pilot_500,tiingo,equity_price_daily,tiingo:ACEVU,ACEVU,2019-01-01,2026-06-02,1,pending,2026-06-02T23:21:24.443361+00:00,2022-11-22,success,1,NaN,2026-06-07 17:48:34.733140+00:00,587,2020-07-28,2022-11-22,0.0,0.0,success,1,NaN,2026-06-07 17:48:34.733140+00:00,True
6,pilot_500:tiingo:equity_price_daily:ADAC:2019-01-01:2026-06-02,pilot_500,tiingo,equity_price_daily,tiingo:ADAC,ADAC,2019-01-01,2026-06-02,1,pending,2026-06-02T23:21:24.443361+00:00,2026-06-02,success,1,NaN,2026-06-07 17:48:35.109349+00:00,461,2019-01-02,2026-06-02,0.0,0.0,success,1,NaN,2026-06-07 17:48:35.109349+00:00,True
7,pilot_500:tiingo:equity_price_daily:AEHAU:2019-01-01:2026-06-02,pilot_500,tiingo,equity_price_daily,tiingo:AEHAU,AEHAU,2019-01-01,2026-06-02,1,pending,2026-06-02T23:21:24.443361+00:00,2023-02-14,success,1,NaN,2026-06-07 17:48:35.512823+00:00,357,2021-09-15,2023-02-14,0.0,0.0,success,1,NaN,2026-06-07 17:48:35.512823+00:00,True
8,pilot_500:tiingo:equity_price_daily:AFCG:2019-01-01:2026-06-02,pilot_500,tiingo,equity_price_daily,tiingo:AFCG,AFCG,2019-01-01,2026-06-02,1,pending,2026-06-02T23:21:24.443361+00:00,2026-06-02,success,1,NaN,2026-06-07 17:48:35.944550+00:00,1307,2021-03-19,2026-06-02,0.0,0.0,success,1,NaN,2026-06-07 17:48:35.944550+00:00,True
9,pilot_500:tiingo:equity_price_daily:AFL:2019-01-01:2026-06-02,pilot_500,tiingo,equity_price_daily,tiingo:AFL,AFL,2019-01-01,2026-06-02,1,pending,2026-06-02T23:21:24.443361+00:00,2026-06-02,success,1,NaN,2026-06-07 17:48:36.390641+00:00,1864,2019-01-02,2026-06-02,0.0,0.0,success,1,NaN,2026-06-07 17:48:36.390641+00:00,True


,dtype
task_id,object
task_list_name,object
source,object
dataset_name,object
security_id,object
ticker,object
requested_start_date,object
requested_end_date,object
priority,int64
task_status,object


## 2. Inspect single Parquet files

In [5]:
parquet_paths = [
    "data/dwd/security_master/dim_security.parquet",
    "data/dwd/security_master/candidate_security_pool.parquet",
    "data/dwd/security_master/backfill_task_list_pilot_500.parquet",
    "data/dwd/security_master/backfill_task_list_bootstrap_candidates.parquet",
]

for p in parquet_paths:
    path_info(p)
    print()


path: c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\security_master\dim_security.parquet
exists: True
size_mb: 3.688

path: c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\security_master\candidate_security_pool.parquet
exists: True
size_mb: 0.289

path: c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\security_master\backfill_task_list_pilot_500.parquet
exists: True
size_mb: 0.019

path: c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\security_master\backfill_task_list_bootstrap_candidates.parquet
exists: True
size_mb: 0.241



In [4]:
parquet_path = PROJECT_ROOT / "data/dwd/equity_price_daily/year=2026/month=06/part-000.parquet"

df = pd.read_parquet(parquet_path)
print(df.head())
df['ticker'].nunique()

  security_id ticker        date    open     high      low   close   volume  adj_open  adj_high  adj_low  adj_close  adj_volume  div_cash  split_factor  source  \
0    tiingo:A      A  2026-06-01  133.50  137.060  132.880  135.98  2678075    133.50   137.060  132.880     135.98     2678075       0.0           1.0  tiingo   
1    tiingo:A      A  2026-06-02  133.28  136.395  132.130  135.05  3021599    133.28   136.395  132.130     135.05     3021599       0.0           1.0  tiingo   
2    tiingo:A      A  2026-06-03  134.22  138.985  133.645  137.40  2928272    134.22   138.985  133.645     137.40     2928272       0.0           1.0  tiingo   
3    tiingo:A      A  2026-06-04  139.50  141.090  137.570  138.37  2126462    139.50   141.090  137.570     138.37     2126462       0.0           1.0  tiingo   
4    tiingo:A      A  2026-06-05  137.95  139.000  134.910  135.44  2201229    137.95   139.000  134.910     135.44     2201229       0.0           1.0  tiingo   

                    l

6379

In [4]:
parquet_path = PROJECT_ROOT / "data/dwd/security_master/candidate_security_pool.parquet"

if parquet_path.exists():
    df_parquet = pd.read_parquet(parquet_path)
    show_df(df_parquet, title="candidate_security_pool.parquet")
    display(df_parquet.dtypes.to_frame("dtype"))

    for col in ["exchange", "asset_type", "price_currency", "is_active"]:
        if col in df_parquet.columns:
            print(f"\n{col} counts")
            display(df_parquet[col].value_counts(dropna=False).head(20).to_frame("n"))



candidate_security_pool.parquet
-------------------------------
shape: (10193, 13)


,security_id,ticker,source_ticker,exchange,asset_type,price_currency,start_date,end_date,is_active,company_name,candidate_pool_name,candidate_reason,loaded_at
0,tiingo:A,A,A,NYSE,Stock,USD,1999-11-18,2026-05-29,True,<NA>,us_common_stock_candidates,asset_type_in_config;currency_in_config;major_us_exchange;overlaps_requested_backfill_window;hyphen_ticker_excluded;...,2026-06-10T16:15:45.503091+00:00
1,tiingo:AA,AA,AA,NYSE,Stock,USD,2016-10-18,2026-05-29,True,<NA>,us_common_stock_candidates,asset_type_in_config;currency_in_config;major_us_exchange;overlaps_requested_backfill_window;hyphen_ticker_excluded;...,2026-06-10T16:15:45.503091+00:00
2,tiingo:AAAC,AAAC,AAAC,NYSE,Stock,USD,2025-12-11,2026-05-29,True,<NA>,us_common_stock_candidates,asset_type_in_config;currency_in_config;major_us_exchange;overlaps_requested_backfill_window;hyphen_ticker_excluded;...,2026-06-10T16:15:45.503091+00:00
3,tiingo:AABA,AABA,AABA,NASDAQ,Stock,USD,1996-04-12,2019-11-06,False,<NA>,us_common_stock_candidates,asset_type_in_config;currency_in_config;major_us_exchange;overlaps_requested_backfill_window;hyphen_ticker_excluded;...,2026-06-10T16:15:45.503091+00:00
4,tiingo:AACB,AACB,AACB,NASDAQ,Stock,USD,2025-04-07,2026-05-29,True,<NA>,us_common_stock_candidates,asset_type_in_config;currency_in_config;major_us_exchange;overlaps_requested_backfill_window;hyphen_ticker_excluded;...,2026-06-10T16:15:45.503091+00:00
5,tiingo:AACBR,AACBR,AACBR,NASDAQ,Stock,USD,2025-04-07,2026-05-29,True,<NA>,us_common_stock_candidates,asset_type_in_config;currency_in_config;major_us_exchange;overlaps_requested_backfill_window;hyphen_ticker_excluded;...,2026-06-10T16:15:45.503091+00:00
6,tiingo:AACBU,AACBU,AACBU,NASDAQ,Stock,USD,2025-02-13,2026-05-29,True,<NA>,us_common_stock_candidates,asset_type_in_config;currency_in_config;major_us_exchange;overlaps_requested_backfill_window;hyphen_ticker_excluded;...,2026-06-10T16:15:45.503091+00:00
7,tiingo:AACG,AACG,AACG,NASDAQ,Stock,USD,2008-01-29,2026-05-29,True,<NA>,us_common_stock_candidates,asset_type_in_config;currency_in_config;major_us_exchange;overlaps_requested_backfill_window;hyphen_ticker_excluded;...,2026-06-10T16:15:45.503091+00:00
8,tiingo:AACI,AACI,AACI,NASDAQ,Stock,USD,2021-11-10,2026-05-29,True,<NA>,us_common_stock_candidates,asset_type_in_config;currency_in_config;major_us_exchange;overlaps_requested_backfill_window;hyphen_ticker_excluded;...,2026-06-10T16:15:45.503091+00:00
9,tiingo:AACIU,AACIU,AACIU,NASDAQ,Stock,USD,2026-02-18,2026-05-29,True,<NA>,us_common_stock_candidates,asset_type_in_config;currency_in_config;major_us_exchange;overlaps_requested_backfill_window;hyphen_ticker_excluded;...,2026-06-10T16:15:45.503091+00:00


,dtype
security_id,string[python]
ticker,string[python]
source_ticker,string[python]
exchange,string[python]
asset_type,string[python]
price_currency,string[python]
start_date,datetime64[ns]
end_date,datetime64[ns]
is_active,bool
company_name,string[python]



exchange counts


,n
exchange,
NASDAQ,7186
NYSE,2916
NYSE MKT,72
NYSE ARCA,19


...

,n
asset_type,
Stock,10193



price_currency counts


,n
price_currency,
USD,10193



is_active counts


,n
is_active,
True,6606
False,3587


## 3. Inspect partitioned DWD Parquet with DuckDB

In [ ]:
from pathlib import Path

cwd = Path.cwd()
print("Current working directory:", cwd)

# Adjust project root robustly
PROJECT_ROOT = cwd
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("PROJECT_ROOT:", PROJECT_ROOT)

dwd_root = PROJECT_ROOT / "data" / "dwd" / "equity_price_daily"

print("DWD root:", dwd_root)
print("DWD root exists:", dwd_root.exists())

if dwd_root.exists():
    print("DWD root children:")
    for p in list(dwd_root.iterdir())[:20]:
        print(" -", p)

files = sorted(dwd_root.glob("**/part-*.parquet"))

print("Matched parquet files:", len(files))
for f in files[:10]:
    print(f)

if not files:
    raise FileNotFoundError(
        f"No part-*.parquet files found under {dwd_root}. "
        "Check whether final DWD output exists or whether notebook PROJECT_ROOT is wrong."
    )

Current working directory: c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\notebook
PROJECT_ROOT: c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform
DWD root: c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\equity_price_daily
DWD root exists: True
DWD root children:
 - c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\equity_price_daily\year=2019
 - c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\equity_price_daily\year=2020
 - c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\equity_price_daily\year=2021
 - c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\equity_price_daily\year=2022
 - c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\equity_price_daily\year=2023
 - c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\equity_price_daily\year=2024
 - c

In [9]:
con = duckdb.connect()

file_paths = [f.as_posix() for f in files]

df = con.read_parquet(file_paths).aggregate("""
    COUNT(*) AS n_rows,
    COUNT(DISTINCT ticker) AS n_tickers,
    MIN(date) AS min_date,
    MAX(date) AS max_date
""").fetchdf()

display(df)

,n_rows,n_tickers,min_date,max_date
0,466001,498,2019-01-02,2026-06-02


In [12]:
df_by_ticker = con.read_parquet(file_paths).query("dwd", """
    SELECT
        ticker,
        COUNT(*) AS n_rows,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM dwd
    GROUP BY ticker
    ORDER BY n_rows desc, ticker
    LIMIT 100
""").fetchdf()

display(df_by_ticker)

,ticker,n_rows,min_date,max_date
0,AAPL,1864,2019-01-02,2026-06-02
1,ACCO,1864,2019-01-02,2026-06-02
2,AFL,1864,2019-01-02,2026-06-02
3,AIG,1864,2019-01-02,2026-06-02
4,ALLO,1864,2019-01-02,2026-06-02
...,...,...,...,...
95,RNR-P-F,1864,2019-01-02,2026-06-02
96,SATS,1864,2019-01-02,2026-06-02
97,SBUX,1864,2019-01-02,2026-06-02
98,SCHL,1864,2019-01-02,2026-06-02


## 4. Inspect local DuckDB database

In [13]:
duckdb_path = PROJECT_ROOT / "data/dbt/quant.duckdb"
path_info(duckdb_path)

if duckdb_path.exists():
    dbt_con = duckdb.connect(str(duckdb_path))
    tables = dbt_con.execute("""
        SELECT table_schema, table_name, table_type
        FROM information_schema.tables
        ORDER BY table_schema, table_name
    """).fetchdf()
    show_df(tables, 100, title="DuckDB tables")
else:
    print("DuckDB dbt database does not exist yet.")


path: c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dbt\quant.duckdb
exists: True
size_mb: 18.762

DuckDB tables
-------------
shape: (5, 3)


,table_schema,table_name,table_type
0,main,ads_ml_research_panel,BASE TABLE
1,main,dws_equity_features_daily,BASE TABLE
2,main,dws_equity_returns_daily,BASE TABLE
3,main,stg_tiingo__equity_price_daily,VIEW
4,main,stg_tiingo_equity_price_daily,VIEW


In [10]:
table_name = "ads_ml_research_panel"

if duckdb_path.exists():
    dbt_con = duckdb.connect(str(duckdb_path))
    try:
        display(dbt_con.execute(f"SELECT COUNT(*) AS n_rows FROM {table_name}").fetchdf())
        sample = dbt_con.execute(f"SELECT * FROM {table_name} LIMIT 20").fetchdf()
        show_df(sample, 20, title=f"Sample rows: {table_name}")
    except Exception as e:
        print(f"Could not query {table_name}:", repr(e))


,n_rows
0,16060



Sample rows: ads_ml_research_panel
----------------------------------
shape: (20, 43)


,security_id,ticker,date,adj_close,adj_volume,volume,div_cash,split_factor,ret_1d,ret_2d,ret_5d,ret_10d,ret_20d,ret_60d,ret_120d,ret_1d_lag1,ret_1d_lag2,ret_5d_lag1,ret_5d_lag2,ret_20d_lag1,dollar_volume,log_dollar_volume,avg_dollar_volume_20d_lagged,avg_dollar_volume_60d_lagged,avg_volume_20d_lagged,dollar_volume_zscore_20d_lagged,volume_zscore_20d_lagged,volatility_20d_lagged,volatility_60d_lagged,max_ret_20d_lagged,min_ret_20d_lagged,close_to_20d_high_lagged,close_to_20d_low_lagged,close_to_60d_high_lagged,close_to_60d_low_lagged,day_of_week,month,fwd_ret_1d,fwd_ret_5d,fwd_ret_20d,label_direction_5d,label_direction_20d,fwd_excess_ret_20d_vs_spy
0,US_SPY,SPY,2020-01-02,296.703989,59037072,59037072,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.751653e+10,23.586411,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,1,-0.007572,0.005479,-0.009665,1,0,0.0
1,US_SPY,SPY,2020-01-03,294.457270,77708081,77708081,0.0,1.0,-0.007572,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.288171e+10,23.853604,1.751653e+10,1.751653e+10,5.903707e+07,NaN,NaN,NaN,NaN,NaN,NaN,-0.007572,-0.007572,-0.007572,-0.007572,5,1,0.003815,0.010235,0.005304,1,1,0.0
2,US_SPY,SPY,2020-01-06,295.580630,55596982,55596982,0.0,1.0,0.003815,-0.003786,NaN,NaN,NaN,NaN,NaN,-0.007572,NaN,NaN,NaN,NaN,1.643339e+10,23.522581,2.019912e+10,2.019912e+10,6.837258e+07,-0.992614,-0.967672,NaN,NaN,-0.007572,-0.007572,-0.003786,0.003815,-0.003786,0.003815,1,1,-0.002812,0.013317,0.016747,1,1,0.0
3,US_SPY,SPY,2020-01-07,294.749526,40461249,40461249,0.0,1.0,-0.002812,0.000993,NaN,NaN,NaN,NaN,NaN,0.003815,-0.007572,NaN,NaN,NaN,1.192593e+10,23.201981,1.894388e+10,1.894388e+10,6.411404e+07,-2.032422,-1.988003,0.008052,0.008052,0.003815,-0.007572,-0.006587,0.000993,-0.006587,0.000993,2,1,0.005330,0.014625,0.031388,1,1,0.0
4,US_SPY,SPY,2020-01-08,296.320403,68177241,68177241,0.0,1.0,0.005330,0.002503,NaN,NaN,NaN,NaN,NaN,-0.002812,0.003815,NaN,NaN,NaN,2.020231e+10,23.729063,1.718939e+10,1.718939e+10,5.820085e+07,0.669344,0.651850,0.005719,0.005719,0.003815,-0.007572,-0.001293,0.006327,-0.001293,0.006327,3,1,0.006781,0.011527,0.029373,1,1,0.0
5,US_SPY,SPY,2020-01-09,298.329665,48421935,48421935,0.0,1.0,0.006781,0.012146,0.005479,NaN,NaN,NaN,NaN,0.005330,-0.002812,NaN,NaN,NaN,1.444570e+10,23.393663,1.779198e+10,1.779198e+10,6.019612e+07,-0.811310,-0.841912,0.005995,0.005995,0.005330,-0.007572,0.005479,0.013151,0.005479,0.013151,4,1,-0.002878,0.013072,0.016991,1,1,0.0
6,US_SPY,SPY,2020-01-10,297.471162,53028660,53028660,0.0,1.0,-0.002878,0.003883,0.010235,NaN,NaN,NaN,NaN,0.006781,0.005330,0.005479,NaN,NaN,1.577450e+10,23.481660,1.723426e+10,1.723426e+10,5.823376e+07,-0.371072,-0.388429,0.006084,0.006084,0.006781,-0.007572,-0.002878,0.010235,-0.002878,0.010235,5,1,0.006877,0.019158,0.027540,1,1,0.0
7,US_SPY,SPY,2020-01-13,299.516955,47026115,47026115,0.0,1.0,0.006877,0.003980,0.013317,NaN,NaN,NaN,NaN,-0.002878,0.006781,0.010235,0.005479,NaN,1.408512e+10,23.368385,1.702572e+10,1.702572e+10,5.749017e+07,-0.809350,-0.844555,0.005679,0.005679,0.006781,-0.007572,0.003980,0.017183,0.003980,0.017183,1,1,-0.001525,0.010215,0.022290,1,1,0.0
8,US_SPY,SPY,2020-01-14,299.060305,62784185,62784185,0.0,1.0,-0.001525,0.005342,0.014625,NaN,NaN,NaN,NaN,0.006877,-0.002878,0.013317,0.010235,NaN,1.877626e+10,23.655859,1.665815e+10,1.665815e+10,5.618217e+07,0.601602,0.547759,0.005726,0.005726,0.006877,-0.007572,-0.001525,0.015632,-0.001525,0.015632,2,1,0.002260,0.011880,0.030447,1,1,0.0
9,US_SPY,SPY,2020-01-15,299.736148,71870739,71870739,0.0,1.0,0.002260,0.000732,0.011527,NaN,NaN,NaN,NaN,-0.001525,0.006877,0.014625,0.013317,NaN,2.154226e+10,23.793282,1.689349e+10,1.689349e+10,5.691572e+07,1.380185,1.301895,0.005399,0.005399,0.006877,-0.007572,0.000732,0.017927,0.000732,0.017927,3,1,0.008318,0.010756,0.027027,1,1,0.0


## 5. Inspect local Postgres metadata

In [14]:
POSTGRES_DSN = os.getenv("POSTGRES_DSN")
print("POSTGRES_DSN exists:", bool(POSTGRES_DSN))
print("psycopg installed:", psycopg is not None)

def postgres_query(sql: str, params: tuple | None = None) -> pd.DataFrame:
    if psycopg is None:
        raise ImportError("psycopg is not installed.")
    if not POSTGRES_DSN:
        raise RuntimeError("POSTGRES_DSN is missing.")
    with psycopg.connect(POSTGRES_DSN) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params or ())
            rows = cur.fetchall()
            columns = [desc.name for desc in cur.description]
    return pd.DataFrame(rows, columns=columns)


POSTGRES_DSN exists: True
psycopg installed: True


In [15]:
try:
    tables_pg = postgres_query("""
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE table_schema = 'metadata'
        ORDER BY table_name
    """)
    show_df(tables_pg, 100, title="Postgres metadata tables")
except Exception as e:
    print("Could not query Postgres:", repr(e))



Postgres metadata tables
------------------------
shape: (4, 2)


,table_schema,table_name
0,metadata,backfill_batches
1,metadata,datasets
2,metadata,pipeline_runs
3,metadata,symbol_ingestion_status


In [16]:
try:
    status_counts = postgres_query("""
        SELECT status, COUNT(*) AS n
        FROM metadata.symbol_ingestion_status
        GROUP BY status
        ORDER BY status
    """)
    show_df(status_counts, 20, title="Symbol ingestion status counts")
except Exception as e:
    print("Could not query symbol_ingestion_status:", repr(e))



Symbol ingestion status counts
------------------------------
shape: (2, 2)


,status,n
0,skipped,2
1,success,498


In [22]:
df = postgres_query("""
    SELECT * 
    FROM metadata.pipeline_runs
""")
show_df(df, 20)

shape: (2, 16)


,run_id,pipeline_name,status,started_at,ended_at,row_count,notes,source,dataset,mode,data_start_date,data_end_date,symbols_count,ods_records,dwd_records,error_message
0,ed094634-5ae4-4ed7-8b82-0ca63fdbe596,transform_tiingo_prices_to_dwd,success,2026-05-21 18:21:00.817079+00:00,2026-05-21 18:21:00.779191+00:00,15080,Transformed Tiingo ODS daily prices to DWD Parquet.,None,None,None,None,None,NaN,NaN,NaN,None
1,90b4442d-590c-4655-8820-0eea2b14b4c7,daily_price_pipeline,success,2026-05-23 01:01:53.128275+00:00,2026-05-23 01:02:13.925367+00:00,16060,"Daily price pipeline completed. Quality result: {'passed': True, 'row_count': 16060, 'symbols_count': 10, 'min_date'...",tiingo,equity_price_daily,incremental,2020-01-01,2026-05-22,10.0,16060.0,16060.0,None


,run_id,pipeline_name,status,started_at,ended_at,row_count,notes,source,dataset,mode,data_start_date,data_end_date,symbols_count,ods_records,dwd_records,error_message
0,ed094634-5ae4-4ed7-8b82-0ca63fdbe596,transform_tiingo_prices_to_dwd,success,2026-05-21 18:21:00.817079+00:00,2026-05-21 18:21:00.779191+00:00,15080,Transformed Tiingo ODS daily prices to DWD Parquet.,None,None,None,None,None,NaN,NaN,NaN,None
1,90b4442d-590c-4655-8820-0eea2b14b4c7,daily_price_pipeline,success,2026-05-23 01:01:53.128275+00:00,2026-05-23 01:02:13.925367+00:00,16060,"Daily price pipeline completed. Quality result: {'passed': True, 'row_count': 16060, 'symbols_count': 10, 'min_date'...",tiingo,equity_price_daily,incremental,2020-01-01,2026-05-22,10.0,16060.0,16060.0,None


In [ ]:
try:
    failed_skipped = postgres_query("""
        SELECT ticker, status, attempt_count, last_successful_date, last_error_message, updated_at
        FROM metadata.symbol_ingestion_status
        WHERE status IN ('failed', 'skipped')
        ORDER BY status, ticker
        LIMIT 100
    """)
    show_df(failed_skipped, 100, title="Failed or skipped symbols")
except Exception as e:
    print("Could not query failed/skipped symbols:", repr(e))


## 6. Inspect GCS bucket

In [13]:
GCS_BUCKET = os.getenv("GCS_BUCKET")
GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID")

print("GCS_BUCKET:", GCS_BUCKET)
print("GCP_PROJECT_ID:", GCP_PROJECT_ID)
print("google-cloud-storage installed:", storage is not None)


GCS_BUCKET: us-quant-data-lake
GCP_PROJECT_ID: nifty-altar-415308
google-cloud-storage installed: True


In [14]:
def list_gcs_objects(prefix: str = "", limit: int = 50) -> pd.DataFrame:
    if storage is None:
        raise ImportError("google-cloud-storage is not installed.")
    if not GCS_BUCKET:
        raise RuntimeError("GCS_BUCKET is missing.")

    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCS_BUCKET)

    rows = []
    for i, blob in enumerate(bucket.list_blobs(prefix=prefix)):
        if i >= limit:
            break
        rows.append({
            "name": blob.name,
            "size_mb": round((blob.size or 0) / 1024 / 1024, 3),
            "updated": blob.updated,
            "content_type": blob.content_type,
        })
    return pd.DataFrame(rows)

try:
    show_df(list_gcs_objects("", 50), 50, title="GCS root object sample")
except Exception as e:
    print("Could not list GCS:", repr(e))



GCS root object sample
----------------------
shape: (50, 4)


,name,size_mb,updated,content_type
0,archive/pre_week6_cleanup_20260601/data/data/dwd/equity_price_daily/year=2020/month=01/part-000.parquet,0.026,2026-06-01 17:34:59.272000+00:00,application/octet-stream
1,archive/pre_week6_cleanup_20260601/data/data/dwd/equity_price_daily/year=2020/month=02/part-000.parquet,0.024,2026-06-01 17:34:59.272000+00:00,application/octet-stream
2,archive/pre_week6_cleanup_20260601/data/data/dwd/equity_price_daily/year=2020/month=03/part-000.parquet,0.026,2026-06-01 17:34:59.271000+00:00,application/octet-stream
3,archive/pre_week6_cleanup_20260601/data/data/dwd/equity_price_daily/year=2020/month=04/part-000.parquet,0.026,2026-06-01 17:35:00.722000+00:00,application/octet-stream
4,archive/pre_week6_cleanup_20260601/data/data/dwd/equity_price_daily/year=2020/month=05/part-000.parquet,0.025,2026-06-01 17:34:59.269000+00:00,application/octet-stream
5,archive/pre_week6_cleanup_20260601/data/data/dwd/equity_price_daily/year=2020/month=06/part-000.parquet,0.026,2026-06-01 17:35:00.722000+00:00,application/octet-stream
6,archive/pre_week6_cleanup_20260601/data/data/dwd/equity_price_daily/year=2020/month=07/part-000.parquet,0.026,2026-06-01 17:35:00.739000+00:00,application/octet-stream
7,archive/pre_week6_cleanup_20260601/data/data/dwd/equity_price_daily/year=2020/month=08/part-000.parquet,0.026,2026-06-01 17:35:00.742000+00:00,application/octet-stream
8,archive/pre_week6_cleanup_20260601/data/data/dwd/equity_price_daily/year=2020/month=09/part-000.parquet,0.026,2026-06-01 17:34:59.644000+00:00,application/octet-stream
9,archive/pre_week6_cleanup_20260601/data/data/dwd/equity_price_daily/year=2020/month=10/part-000.parquet,0.027,2026-06-01 17:35:00.725000+00:00,application/octet-stream


In [15]:
for prefix in [
    "ods/source=tiingo/dataset=supported_tickers/",
    "ods/source=tiingo/dataset=equity_price_daily/",
    "dwd/security_master/",
    "dwd/equity_price_daily/",
    "reports/backfill_audit/",
]:
    print("\nPREFIX:", prefix)
    try:
        display(list_gcs_objects(prefix, 20))
    except Exception as e:
        print("Could not list prefix:", repr(e))



PREFIX: ods/source=tiingo/dataset=supported_tickers/


,name,size_mb,updated,content_type
0,ods/source=tiingo/dataset=supported_tickers/supported_tickers.csv,5.236,2026-06-01 14:17:49.989000+00:00,application/vnd.ms-excel



PREFIX: ods/source=tiingo/dataset=equity_price_daily/


,name,size_mb,updated,content_type
0,ods/source=tiingo/dataset=equity_price_daily/symbol=AACIU/aaciu_prices.json,0.022,2026-06-08 15:20:26.389000+00:00,application/json
1,ods/source=tiingo/dataset=equity_price_daily/symbol=AAPL/aapl_prices.json,0.641,2026-06-08 15:20:26.703000+00:00,application/json
2,ods/source=tiingo/dataset=equity_price_daily/symbol=ABOS/abos_prices.json,0.366,2026-06-08 15:20:26.973000+00:00,application/json
3,ods/source=tiingo/dataset=equity_price_daily/symbol=ACACU/acacu_prices.json,0.169,2026-06-08 15:20:27.175000+00:00,application/json
4,ods/source=tiingo/dataset=equity_price_daily/symbol=ACCO/acco_prices.json,0.608,2026-06-08 15:20:27.459000+00:00,application/json
5,ods/source=tiingo/dataset=equity_price_daily/symbol=ACEVU/acevu_prices.json,0.175,2026-06-08 15:20:27.664000+00:00,application/json
6,ods/source=tiingo/dataset=equity_price_daily/symbol=ADAC/adac_prices.json,0.140,2026-06-08 15:20:27.872000+00:00,application/json
7,ods/source=tiingo/dataset=equity_price_daily/symbol=AEHAU/aehau_prices.json,0.106,2026-06-08 15:20:28.075000+00:00,application/json
8,ods/source=tiingo/dataset=equity_price_daily/symbol=AFCG/afcg_prices.json,0.427,2026-06-08 15:20:28.359000+00:00,application/json
9,ods/source=tiingo/dataset=equity_price_daily/symbol=AFL/afl_prices.json,0.628,2026-06-08 15:20:28.704000+00:00,application/json



PREFIX: dwd/security_master/


,name,size_mb,updated,content_type
0,dwd/security_master/backfill_task_list_bootstrap_candidates.parquet,0.241,2026-06-08 19:23:03.823000+00:00,application/octet-stream
1,dwd/security_master/backfill_task_list_pilot_500.parquet,0.019,2026-06-02 23:21:26.690000+00:00,application/octet-stream
2,dwd/security_master/candidate_security_pool.parquet,0.289,2026-06-08 19:22:37.051000+00:00,application/octet-stream
3,dwd/security_master/dim_security.parquet,3.688,2026-06-02 18:52:24.199000+00:00,application/octet-stream



PREFIX: dwd/equity_price_daily/


,name,size_mb,updated,content_type
0,dwd/equity_price_daily/year=2019/month=01/part-000.parquet,0.273,2026-06-08 15:22:20.194000+00:00,application/octet-stream
1,dwd/equity_price_daily/year=2019/month=02/part-000.parquet,0.248,2026-06-08 15:22:20.383000+00:00,application/octet-stream
2,dwd/equity_price_daily/year=2019/month=03/part-000.parquet,0.274,2026-06-08 15:22:20.591000+00:00,application/octet-stream
3,dwd/equity_price_daily/year=2019/month=04/part-000.parquet,0.275,2026-06-08 15:22:20.803000+00:00,application/octet-stream
4,dwd/equity_price_daily/year=2019/month=05/part-000.parquet,0.289,2026-06-08 15:22:21.050000+00:00,application/octet-stream
5,dwd/equity_price_daily/year=2019/month=06/part-000.parquet,0.269,2026-06-08 15:22:21.236000+00:00,application/octet-stream
6,dwd/equity_price_daily/year=2019/month=07/part-000.parquet,0.286,2026-06-08 15:22:21.431000+00:00,application/octet-stream
7,dwd/equity_price_daily/year=2019/month=08/part-000.parquet,0.286,2026-06-08 15:22:21.614000+00:00,application/octet-stream
8,dwd/equity_price_daily/year=2019/month=09/part-000.parquet,0.263,2026-06-08 15:22:21.800000+00:00,application/octet-stream
9,dwd/equity_price_daily/year=2019/month=10/part-000.parquet,0.293,2026-06-08 15:22:21.997000+00:00,application/octet-stream



PREFIX: reports/backfill_audit/


""


In [ ]:
def read_gcs_text_object(object_name: str) -> str:
    if storage is None:
        raise ImportError("google-cloud-storage is not installed.")
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCS_BUCKET)
    blob = bucket.blob(object_name)
    if not blob.exists():
        raise FileNotFoundError(f"Object not found: gs://{GCS_BUCKET}/{object_name}")
    return blob.download_as_text()

gcs_supported_tickers = "ods/source=tiingo/dataset=supported_tickers/supported_tickers.csv"

try:
    text = read_gcs_text_object(gcs_supported_tickers)
    df_gcs_csv = pd.read_csv(StringIO(text))
    show_df(df_gcs_csv, 10, title="GCS supported_tickers.csv")
except Exception as e:
    print("Could not read GCS CSV:", repr(e))


In [ ]:
def download_gcs_object_to_local(object_name: str, local_path: str | Path) -> Path:
    if storage is None:
        raise ImportError("google-cloud-storage is not installed.")
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCS_BUCKET)
    blob = bucket.blob(object_name)
    if not blob.exists():
        raise FileNotFoundError(f"Object not found: gs://{GCS_BUCKET}/{object_name}")
    blob.download_to_filename(str(local_path))
    return local_path

gcs_candidate_pool = "dwd/security_master/candidate_security_pool.parquet"
local_tmp = PROJECT_ROOT / "data/_preview/gcs_candidate_security_pool.parquet"

try:
    local_file = download_gcs_object_to_local(gcs_candidate_pool, local_tmp)
    df_gcs_parquet = pd.read_parquet(local_file)
    show_df(df_gcs_parquet, 10, title="Downloaded GCS candidate_security_pool.parquet")
except Exception as e:
    print("Could not download/read GCS parquet:", repr(e))


## 7. Quick health checklist

In [ ]:
checks = {
    "supported_tickers_csv_local": PROJECT_ROOT / "data/ods/source=tiingo/dataset=supported_tickers/supported_tickers.csv",
    "dim_security_local": PROJECT_ROOT / "data/dwd/security_master/dim_security.parquet",
    "candidate_pool_local": PROJECT_ROOT / "data/dwd/security_master/candidate_security_pool.parquet",
    "pilot_task_list_local": PROJECT_ROOT / "data/dwd/security_master/backfill_task_list_pilot_500.parquet",
    "bootstrap_task_list_local": PROJECT_ROOT / "data/dwd/security_master/backfill_task_list_bootstrap_candidates.parquet",
    "dwd_price_root_local": PROJECT_ROOT / "data/dwd/equity_price_daily",
    "audit_report_root_local": PROJECT_ROOT / "reports/backfill_audit",
    "duckdb_dbt_local": PROJECT_ROOT / "data/dbt/quant.duckdb",
}

health = pd.DataFrame([
    {
        "name": name,
        "path": str(path),
        "exists": path.exists(),
        "is_file": path.is_file(),
        "is_dir": path.is_dir(),
    }
    for name, path in checks.items()
])

display(health)


,name,path,exists,is_file,is_dir
0,supported_tickers_csv_local,c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\ods\source=tiingo\dataset=supported_ticker...,True,True,False
1,dim_security_local,c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\security_master\dim_security.parquet,True,True,False
2,candidate_pool_local,c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\security_master\candidate_security_poo...,True,True,False
3,pilot_task_list_local,c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\security_master\backfill_task_list_pil...,True,True,False
4,bootstrap_task_list_local,c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\security_master\backfill_task_list_boo...,True,True,False
5,dwd_price_root_local,c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dwd\equity_price_daily,True,False,True
6,audit_report_root_local,c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\reports\backfill_audit,True,False,True
7,duckdb_dbt_local,c:\Users\elisa\Desktop\git\codes\quant\us-equity-quant-data-platform\data\dbt\quant.duckdb,True,True,False


: 

## Recommended workflow

```text
CSV / small samples       -> pandas or Data Wrangler
single Parquet files      -> pandas
partitioned Parquet       -> DuckDB
metadata                  -> Postgres SQL
cloud file inventory      -> GCS list_blobs
dashboard / large query   -> BigQuery / Looker
```
